In [1]:
import folium
from folium.plugins import MarkerCluster, TimestampedGeoJson
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. СИНТЕТИЧНІ ДАНІ ПО ВСІЙ ТЕРИТОРІЇ УКРАЇНИ
# ---------------------------------------------------------
np.random.seed(42)

# Опорні вузлові райони в Україні
regions = {
    "Харківський напрямок": (49.9935, 36.2304),
    "Куп'янський сектор": (49.7075, 37.6167),
    "Лиманський сектор": (48.9850, 37.8100),
    "Бахмутський сектор": (48.5987, 37.9981),
    "Запорізький напрямок": (47.8388, 35.1396),
    "Херсонський сектор": (46.6354, 32.6169),
    "Сумський напрямок": (50.9077, 34.7981),
    "Одеський район": (46.4825, 30.7233),
    "Київський вузол": (50.4501, 30.5234),
    "Дніпровський вузол": (48.4647, 35.0462),
}

event_types = ["Артилерійський обстріл", "Удар FPV-дрона", "Стрілецький бій", "Авіаудар", "Ракетний удар"]
intensity_levels = ["Низька", "Середня", "Висока", "Критична"]

n_events = 50
timestamps = pd.date_range(start="2026-10-23 00:00:00", periods=n_events, freq="30min")

chosen_regions = np.random.choice(list(regions.keys()), size=n_events)
lats = [regions[r][0] + np.random.normal(0, 0.15) for r in chosen_regions]
lons = [regions[r][1] + np.random.normal(0, 0.15) for r in chosen_regions]

df_events = pd.DataFrame({
    "event_id": [f"UA-EVT-{2000 + i}" for i in range(n_events)],
    "timestamp": timestamps.strftime("%Y-%m-%d %H:%M:%S"),
    "region": chosen_regions,
    "latitude": lats,
    "longitude": lons,
    "event_type": np.random.choice(event_types, size=n_events, p=[0.35, 0.30, 0.15, 0.10, 0.10]),
    "intensity": np.random.choice(intensity_levels, size=n_events, p=[0.2, 0.4, 0.3, 0.1]),
    "damage_reported": np.random.choice(["Ні", "Частково", "Так"], size=n_events, p=[0.4, 0.4, 0.2]),
})

df_events.to_csv("tactical_events_ukraine.csv", index=False, encoding="utf-8")
print("Файл тактичних подій збережено: tactical_events_ukraine.csv")

# ---------------------------------------------------------
# 2. НАЛАШТУВАННЯ ВІЙСЬКОВО-ТАКТИЧНИХ МАРКЕРІВ
# ---------------------------------------------------------
color_map = {
    "Артилерійський обстріл": "orange",
    "Удар FPV-дрона": "red",
    "Стрілецький бій": "blue",
    "Авіаудар": "darkred",
    "Ракетний удар": "purple",
}

icon_map = {
    "Артилерійський обстріл": "crosshairs",
    "Удар FPV-дрона": "plane",
    "Стрілецький бій": "shield-halved",
    "Авіаудар": "bomb",
    "Ракетний удар": "rocket",
}

# ---------------------------------------------------------
# 3. ПОБУДОВА КАРТИ УКРАЇНИ
# ---------------------------------------------------------
# Центр України: Добровеличківка / Кропивницький район, масштаб 6 охоплює всі кордони
ukraine_map = folium.Map(
    location=[48.5, 31.5],
    zoom_start=6,
    tiles="OpenStreetMap",  # Стандартна детальна мапа з містами України українською мовою
    control_scale=True,
)

# Шар 1: Кластери подій по областях/секторах
cluster_group = folium.FeatureGroup(name="Осередки активності (Кластери)")
marker_cluster = MarkerCluster().add_to(cluster_group)

for _, row in df_events.iterrows():
    popup_html = f"""
    <div style="font-family: sans-serif; font-size: 12px; width: 220px;">
        <b style="color: #0b5394;">{row['event_id']}</b><br>
        <b>Район:</b> {row['region']}<br>
        <b>Час:</b> {row['timestamp']}<br>
        <b>Тип:</b> {row['event_type']}<br>
        <b>Рівень:</b> {row['intensity']}<br>
        <b>Пошкодження:</b> {row['damage_reported']}
    </div>
    """
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{row['event_type']} — {row['region']}",
        icon=folium.Icon(
            color=color_map.get(row["event_type"], "gray"),
            icon=icon_map.get(row["event_type"], "info-sign"),
            prefix="fa",
        ),
    ).add_to(marker_cluster)

cluster_group.add_to(ukraine_map)

# Шар 2: Часовий повзунок (динаміка по Україні)
features = []
for _, row in df_events.iterrows():
    iso_time = pd.to_datetime(row["timestamp"]).isoformat()
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row["longitude"], row["latitude"]],
        },
        "properties": {
            "time": iso_time,
            "style": {"color": color_map.get(row["event_type"], "red")},
            "icon": "circle",
            "iconstyle": {
                "fillColor": color_map.get(row["event_type"], "red"),
                "fillOpacity": 0.85,
                "stroke": "true",
                "radius": 8,
            },
            "popup": f"<b>{row['event_type']}</b> ({row['region']})<br>{row['timestamp']}",
        },
    }
    features.append(feature)

geojson_data = {"type": "FeatureCollection", "features": features}

time_slider = TimestampedGeoJson(
    geojson_data,
    period="PT30M",
    duration="PT1H",
    auto_play=False,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm:ss",
    time_slider_drag_update=True,
)
time_slider.add_to(ukraine_map)

folium.LayerControl(collapsed=False).add_to(ukraine_map)

# Збереження та вивід
output_html = "ukraine_tactical_map.html"
ukraine_map.save(output_html)
print(f"Карту України збережено: {output_html}")

ukraine_map

Файл тактичних подій збережено: tactical_events_ukraine.csv
Карту України збережено: ukraine_tactical_map.html
